# Multi-asset portfolio backtest

**Hypothetical client:** age 40, moderate risk tolerance, 15+ year investment horizon, no near-term withdrawals. Initial investment: $10,000. The notebook compares a diversified allocation with a 60/40 VTI–BND benchmark over January 2011–December 2025.

Run cells from top to bottom. Install `yfinance` in the notebook environment first if needed: `%pip install yfinance`. The notebook downloads adjusted historical prices with `yfinance`.

## 1. Allocation and setup

| Sleeve | Proxy | Weight | Rationale |
|---|---|---:|---|
| US equity | VTI | 40% | Broad US exposure |
| Developed ex-US equity | VEA | 15% | Geographic diversification |
| Investment-grade bonds | BND | 25% | Fixed income |
| Gold | GLD | 10% | Alternative exposure |
| Listed real estate | VNQ | 5% | REIT exposure |
| Short Treasuries | SHY | 5% | Cash-like proxy |

The portfolio and benchmark are rebalanced monthly. SHY is a short Treasury ETF, not cash in a bank account.

In [ ]:
import csv
import math
from pathlib import Path
import statistics
import yfinance as yf

ROOT = Path.cwd()
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
ASSETS = {'VTI': .40, 'VEA': .15, 'BND': .25, 'GLD': .10, 'VNQ': .05, 'SHY': .05}
BENCH = {'VTI': .60, 'BND': .40}
TICKERS = list(ASSETS)
START = '2010-12-01'
END = '2026-01-01'
assert abs(sum(ASSETS.values()) - 1) < 1e-12

## 2. Load adjusted monthly prices

[`yfinance.download`](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html) retrieves the six ETFs together. `auto_adjust=True` makes `Close` adjusted for the provider's corporate actions. The data can change if Yahoo revises its history. The first download needs internet access.

In [ ]:
def metrics(returns):
    n = len(returns)
    growth = math.prod(1 + x for x in returns)
    peak = value = 1.0
    worst_dd = 0.0
    for r in returns:
        value *= 1 + r
        peak = max(peak, value)
        worst_dd = min(worst_dd, value / peak - 1)
    annual = growth ** (12 / n) - 1
    vol = statistics.stdev(returns) * math.sqrt(12)
    return {"ending_10000": 10000 * growth, "cagr": annual, "volatility": vol,
            "sharpe_zero_rf": statistics.mean(returns) * 12 / vol,
            "max_drawdown": worst_dd, "positive_months": sum(r > 0 for r in returns) / n}

In [ ]:
download = yf.download(
    TICKERS, start=START, end=END, interval='1mo',
    auto_adjust=True, progress=False, group_by='ticker',
    multi_level_index=True, threads=False,
)
price = {}
for ticker in TICKERS:
    series = download[ticker]['Close'].dropna()
    price[ticker] = {date.strftime('%Y-%m'): float(value)
                     for date, value in series.items()}

months = sorted(set.intersection(*(set(series) for series in price.values())))
months = [m for m in months if '2010-12' <= m <= '2025-12']
assert months[0] == '2010-12' and months[-1] == '2025-12' and len(months) == 181
print(f'{len(months)-1} monthly returns: {months[1]} through {months[-1]}')

## 3. Calculate monthly returns and wealth

Each month’s portfolio return is the weighted sum of its ETF returns. Applying the target weights each month models monthly rebalancing without look-ahead.

In [ ]:
rows, p_rets, b_rets = [], [], []
p_val = b_val = 10000.0
for prev, month in zip(months, months[1:]):
    asset_ret = {t: price[t][month] / price[t][prev] - 1 for t in TICKERS}
    pr = sum(w * asset_ret[t] for t, w in ASSETS.items())
    br = sum(w * asset_ret[t] for t, w in BENCH.items())
    p_val *= 1 + pr
    b_val *= 1 + br
    p_rets.append(pr)
    b_rets.append(br)
    rows.append({'month': month, **{f'{t}_adjusted_close': price[t][month] for t in TICKERS},
                 **{f'{t}_return': asset_ret[t] for t in TICKERS},
                 'portfolio_return': pr, 'benchmark_return': br,
                 'portfolio_value': p_val, 'benchmark_value': b_val})

with (OUT / 'monthly_backtest.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print('Saved', OUT / 'monthly_backtest.csv')


## 4. Risk and return summary

CAGR measures compounded growth. Volatility is the sample standard deviation of monthly returns times √12. Maximum drawdown uses month-end wealth, so intramonth losses may be larger. The displayed Sharpe-style ratio assumes a 0% risk-free rate for comparability; it is not an excess-return estimate against historical Treasury bills.

In [ ]:
pm, bm = metrics(p_rets), metrics(b_rets)
formats = [('Ending value of $10,000', 'ending_10000', '${:,.0f}'),
           ('CAGR', 'cagr', '{:.2%}'), ('Annualized volatility', 'volatility', '{:.2%}'),
           ('Sharpe (0% risk-free)', 'sharpe_zero_rf', '{:.2f}'),
           ('Maximum month-end drawdown', 'max_drawdown', '{:.2%}'),
           ('Positive months', 'positive_months', '{:.1%}')]
print(f"{'Metric':32} {'Multi-asset':>15} {'60/40':>15}")
for label, key, fmt in formats:
    print(f'{label:32} {fmt.format(pm[key]):>15} {fmt.format(bm[key]):>15}')


## 5. Market cycles and annual returns

The annual table makes it easier to discuss the 2020 shock, the 2021 rebound, and the 2022 stock/bond selloff.

In [ ]:
yearly = []
for year in range(2011, 2026):
    subset = [r for r in rows if r['month'].startswith(str(year))]
    yearly.append((year, math.prod(1+r['portfolio_return'] for r in subset)-1,
                   math.prod(1+r['benchmark_return'] for r in subset)-1))
with (OUT / 'annual_returns.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['year', 'portfolio', 'benchmark_60_40'])
    writer.writerows(yearly)
print(f"{'Year':>4} {'Multi-asset':>14} {'60/40':>14}")
for y, p, b in yearly:
    print(f'{y:>4} {p:>13.2%} {b:>13.2%}')


## 6. Interpretation and limits

The multi-asset mix should be judged on return **and** risk, including volatility, drawdown, and behavior in difficult years. Diversification does not guarantee lower losses: assets can become more correlated during stress. This is a hypothetical, retrospective result, not a forecast. It omits fees, taxes, trading costs, cash flows, and inflation. The selected ETFs and time period influence every result.